In [ ]:
!pip install xgboost catboost optuna

In [ ]:
# 1. 라이브러리 및 데이터 로드
import pandas as pd
import numpy as np
import warnings
import os
import joblib
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

train_path = '/content/drive/MyDrive/LG aimers/open/train.csv'
test_path = '/content/drive/MyDrive/LG aimers/open/test.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("데이터 로드 완료:", train.shape)


In [ ]:
# 2. 기초 범주형 변수 인코딩
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold

label_encoders = {}
categorical_cols = ['top_bottom', 'pitcher_hand', 'batter_hand', 'game_type', 'base_state']

for col in categorical_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    label_encoders[col] = le


In [ ]:
# 3. 근본적 전처리 대공사 (Phase 6: 영혼까지 끌어모으기)

# (1) K-Fold Target Encoding (선수 고유 ID 활용)
skf_te = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train['pitcher_id_te'] = np.nan
train['batter_id_te'] = np.nan

global_pitcher_mean = train.groupby('pitcher_id')['control_success'].mean().to_dict()
global_batter_mean = train.groupby('batter_id')['control_success'].mean().to_dict()
global_all_mean = train['control_success'].mean()

for train_idx, val_idx in skf_te.split(train, train['control_success']):
    X_tr, X_va = train.iloc[train_idx], train.iloc[val_idx]
    
    p_mean = X_tr.groupby('pitcher_id')['control_success'].mean()
    b_mean = X_tr.groupby('batter_id')['control_success'].mean()
    
    train.loc[val_idx, 'pitcher_id_te'] = X_va['pitcher_id'].map(p_mean)
    train.loc[val_idx, 'batter_id_te'] = X_va['batter_id'].map(b_mean)

train['pitcher_id_te'] = train['pitcher_id_te'].fillna(train['pitcher_id'].map(global_pitcher_mean)).fillna(global_all_mean)
train['batter_id_te'] = train['batter_id_te'].fillna(train['batter_id'].map(global_batter_mean)).fillna(global_all_mean)


# (2) 비율 데이터의 압축 해제 (카운트 분해)
train['pitcher_success_count'] = train['asof_pitcher_n'] * train['asof_pitcher_success_rate']
train['pitcher_fail_count'] = train['asof_pitcher_n'] - train['pitcher_success_count']
train['batter_success_count'] = train['asof_batter_n'] * train['asof_batter_success_rate']
train['batter_fail_count'] = train['asof_batter_n'] - train['batter_success_count']

# (3) 교차 상호작용 피처 (Cross-Interaction)
train['pitcher_vs_batter_exp'] = train['asof_pitcher_n'] / (train['asof_batter_n'] + 1)
train['is_pitcher_behind'] = (train['balls_before'] > train['strikes_before']).astype(int)
train['is_full_count'] = ((train['balls_before'] == 3) & (train['strikes_before'] == 2)).astype(int)
train['is_clutch'] = ((train['inning'] >= 7) & (train['score_diff_home'].abs() <= 1)).astype(int)
train['is_garbage_time'] = (train['score_diff_home'].abs() >= 7).astype(int)

train['experience_diff'] = train['asof_pitcher_n'] - train['asof_batter_n']
train['success_rate_diff'] = train['asof_pitcher_success_rate'] - train['asof_batter_success_rate']
train['count_pressure_ratio'] = train['balls_before'] / (train['strikes_before'] + 1)
train['pressure_vs_success'] = train['count_pressure_ratio'] * train['asof_pitcher_success_rate']
train['fatigue_proxy'] = train['inning'] * train['outs_before']
train['granular_fatigue'] = (train['inning'] * 15) + (train['outs_before'] * 5) + train['balls_before'] + train['strikes_before']
train['clutch_leverage'] = train['is_clutch'] * train['li']

# (4) 누수 없는 스무딩 인코딩 (Smoothed Target Encoding)
def smooth_target_encoding(df, col, target_col, weight=50):
    global_mean = df[target_col].mean()
    agg = df.groupby(col)[target_col].agg(['count', 'mean'])
    counts = agg['count']
    means = agg['mean']
    smooth = (counts * means + weight * global_mean) / (counts + weight)
    return smooth.to_dict(), global_mean

target_means = {}
global_means = {}

train['platoon'] = train['pitcher_hand'].astype(str) + "_" + train['batter_hand'].astype(str)
train['clutch_platoon'] = train['is_clutch'].astype(str) + "_" + train['platoon']

for col in ['game_type', 'base_state', 'inning', 'platoon', 'clutch_platoon']:
    mapping, g_mean = smooth_target_encoding(train, col, 'control_success', weight=50)
    target_means[col] = mapping
    global_means[col] = g_mean
    train[f'{col}_target_enc'] = train[col].map(mapping)

print("근본적 전처리(Phase 6) 파생 변수 및 인코딩 완료.")


In [ ]:
# 4. 최종 피처 리스트 정리
base_features = ['li', 'pitcher_hand', 'batter_hand']
asof_cols = [col for col in train.columns if 'asof_' in col]
base_features.extend(asof_cols)

derived_features = [
    'pitcher_id_te', 'batter_id_te', # 강력한 고유 ID 효과
    'pitcher_success_count', 'pitcher_fail_count', 'batter_success_count', 'batter_fail_count',
    'pitcher_vs_batter_exp', 'pressure_vs_success',
    'is_pitcher_behind', 'is_full_count', 'is_clutch', 'is_garbage_time', 
    'experience_diff', 'success_rate_diff',
    'count_pressure_ratio', 'fatigue_proxy', 'clutch_leverage', 'granular_fatigue',
    'game_type_target_enc', 'base_state_target_enc', 'inning_target_enc', 
    'platoon_target_enc', 'clutch_platoon_target_enc'
]

final_features = base_features + derived_features
print(f"Total features count: {len(final_features)}")


In [ ]:
# 5. 3대장 멀티 모델 앙상블 및 Isotonic 보정 (과적합 없는 안정적 파라미터 적용)
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import mean_squared_error

X = train[final_features]
y = train['control_success']

# 안정성 및 고속화가 보장된 파라미터
best_lgb_params = {
    'objective': 'binary', 'learning_rate': 0.03, 'num_leaves': 64, 'max_depth': 8,
    'subsample': 0.8, 'colsample_bytree': 0.8, 'n_estimators': 1500, 'verbosity': -1, 'random_state': 42
}

best_xgb_params = {
    'objective': 'binary:logistic', 'eval_metric': 'rmse', 'learning_rate': 0.03, 'max_depth': 7,
    'subsample': 0.8, 'colsample_bytree': 0.8, 'n_estimators': 1500, 'tree_method': 'hist', 'random_state': 42
}

best_cb_params = {
    'loss_function': 'Logloss', 'learning_rate': 0.03, 'depth': 6, 'iterations': 1500, 'random_seed': 42, 'verbose': 0
}

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

calibrated_lgb_models = []
calibrated_xgb_models = []
calibrated_cb_models = []
oof_preds = np.zeros(len(train))

print("5-Fold 앙상블 학습 시작...")
for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[valid_idx], y.iloc[valid_idx]
    
    print(f"=== Fold {fold+1} ===")
    
    # 1. LightGBM
    lgb_model = lgb.LGBMClassifier(**best_lgb_params)
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='mse', callbacks=[lgb.early_stopping(50, verbose=False)])
    calib_lgb = CalibratedClassifierCV(estimator=lgb_model, method='isotonic', cv='prefit')
    calib_lgb.fit(X_val, y_val)
    calibrated_lgb_models.append(calib_lgb)
    
    # 2. XGBoost
    xgb_model = xgb.XGBClassifier(**best_xgb_params)
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    calib_xgb = CalibratedClassifierCV(estimator=xgb_model, method='isotonic', cv='prefit')
    calib_xgb.fit(X_val, y_val)
    calibrated_xgb_models.append(calib_xgb)
    
    # 3. CatBoost
    cb_model = cb.CatBoostClassifier(**best_cb_params)
    cb_model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False, early_stopping_rounds=50)
    calib_cb = CalibratedClassifierCV(estimator=cb_model, method='isotonic', cv='prefit')
    calib_cb.fit(X_val, y_val)
    calibrated_cb_models.append(calib_cb)
    
    # 앙상블 가중평균
    fold_preds = (calib_lgb.predict_proba(X_val)[:, 1] * 0.35) + \
                 (calib_xgb.predict_proba(X_val)[:, 1] * 0.35) + \
                 (calib_cb.predict_proba(X_val)[:, 1] * 0.30)
    oof_preds[valid_idx] = fold_preds
    
    print(f"-> Fold {fold+1} Brier Score: {mean_squared_error(y_val, fold_preds):.5f}")

print(f"\n최종 Out-Of-Fold Brier Score: {mean_squared_error(y, oof_preds):.5f}")


In [ ]:
# 6. 추론 스크립트 작성 및 제출용 압축파일 생성
import os
import joblib
import zipfile

os.makedirs('./submission/model', exist_ok=True)

# 모델 추출 (C++ 코어 직접 추출로 버전 에러 방지)
for i, calibrator in enumerate(calibrated_lgb_models):
    calibrator.estimator.booster_.save_model(f'./submission/model/lgb_{i}.txt')
    joblib.dump(calibrator.calibrated_classifiers_[0].calibrators[0], f'./submission/model/iso_lgb_{i}.pkl')

for i, calibrator in enumerate(calibrated_xgb_models):
    calibrator.estimator.save_model(f'./submission/model/xgb_{i}.json')
    joblib.dump(calibrator.calibrated_classifiers_[0].calibrators[0], f'./submission/model/iso_xgb_{i}.pkl')

for i, calibrator in enumerate(calibrated_cb_models):
    calibrator.estimator.save_model(f'./submission/model/cb_{i}.cbm')
    joblib.dump(calibrator.calibrated_classifiers_[0].calibrators[0], f'./submission/model/iso_cb_{i}.pkl')

joblib.dump(label_encoders, './submission/model/encoders.pkl')
joblib.dump(final_features, './submission/model/final_features.pkl')
joblib.dump(target_means, './submission/model/target_means.pkl')
joblib.dump(global_means, './submission/model/global_means.pkl')

# Test ID Encoding 저장을 위한 딕셔너리
test_id_mappings = {
    'global_pitcher_mean': global_pitcher_mean,
    'global_batter_mean': global_batter_mean,
    'global_all_mean': global_all_mean
}
joblib.dump(test_id_mappings, './submission/model/test_id_mappings.pkl')

with open('./submission/requirements.txt', 'w') as f:
    f.write("pandas\nscikit-learn\nlightgbm\nxgboost\ncatboost\njoblib\n")

script_code = """import os
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import joblib

def main():
    TEST_PATH = "./data/test.csv"
    SUB_PATH = "./data/sample_submission.csv"
    
    label_encoders = joblib.load("./model/encoders.pkl")
    final_features = joblib.load("./model/final_features.pkl")
    target_means = joblib.load("./model/target_means.pkl")
    global_means = joblib.load("./model/global_means.pkl")
    test_id_mappings = joblib.load("./model/test_id_mappings.pkl")
        
    test = pd.read_csv(TEST_PATH)
    sub = pd.read_csv(SUB_PATH)
    
    # 1. Label Encoding
    for col in ['top_bottom', 'pitcher_hand', 'batter_hand', 'game_type', 'base_state']:
        le = label_encoders[col]
        test[col] = test[col].astype(str).apply(lambda x: x if x in le.classes_ else le.classes_[0])
        test[col] = le.transform(test[col])
        
    # 2. 고유 ID Target Encoding
    test['pitcher_id_te'] = test['pitcher_id'].map(test_id_mappings['global_pitcher_mean']).fillna(test_id_mappings['global_all_mean'])
    test['batter_id_te'] = test['batter_id'].map(test_id_mappings['global_batter_mean']).fillna(test_id_mappings['global_all_mean'])
    
    # 3. Categorical Smoothed Target Encoding
    test['platoon'] = test['pitcher_hand'].astype(str) + "_" + test['batter_hand'].astype(str)
    test['is_clutch'] = ((test['inning'] >= 7) & (test['score_diff_home'].abs() <= 1)).astype(int)
    test['clutch_platoon'] = test['is_clutch'].astype(str) + "_" + test['platoon']
    
    for col in ['game_type', 'base_state', 'inning', 'platoon', 'clutch_platoon']:
        test[f'{col}_target_enc'] = test[col].map(target_means[col]).fillna(global_means[col])
        
    # 4. 압축 해제 및 상호작용
    test['pitcher_success_count'] = test['asof_pitcher_n'] * test['asof_pitcher_success_rate']
    test['pitcher_fail_count'] = test['asof_pitcher_n'] - test['pitcher_success_count']
    test['batter_success_count'] = test['asof_batter_n'] * test['asof_batter_success_rate']
    test['batter_fail_count'] = test['asof_batter_n'] - test['batter_success_count']
    
    test['pitcher_vs_batter_exp'] = test['asof_pitcher_n'] / (test['asof_batter_n'] + 1)
    test['is_pitcher_behind'] = (test['balls_before'] > test['strikes_before']).astype(int)
    test['is_full_count'] = ((test['balls_before'] == 3) & (test['strikes_before'] == 2)).astype(int)
    test['is_garbage_time'] = (test['score_diff_home'].abs() >= 7).astype(int)
    
    test['experience_diff'] = test['asof_pitcher_n'] - test['asof_batter_n']
    test['success_rate_diff'] = test['asof_pitcher_success_rate'] - test['asof_batter_success_rate']
    test['count_pressure_ratio'] = test['balls_before'] / (test['strikes_before'] + 1)
    test['pressure_vs_success'] = test['count_pressure_ratio'] * test['asof_pitcher_success_rate']
    test['fatigue_proxy'] = test['inning'] * test['outs_before']
    test['granular_fatigue'] = (test['inning'] * 15) + (test['outs_before'] * 5) + test['balls_before'] + test['strikes_before']
    test['clutch_leverage'] = test['is_clutch'] * test['li']
    
    # 5. 모델 앙상블 추론
    X_test = test[final_features]
    final_preds = np.zeros(len(test))
    
    for i in range(5):
        lgb_booster = lgb.Booster(model_file=f'./model/lgb_{i}.txt')
        iso_lgb = joblib.load(f'./model/iso_lgb_{i}.pkl')
        raw_lgb = lgb_booster.predict(X_test)
        
        xgb_model = xgb.XGBClassifier()
        xgb_model.load_model(f'./model/xgb_{i}.json')
        iso_xgb = joblib.load(f'./model/iso_xgb_{i}.pkl')
        raw_xgb = xgb_model.predict_proba(X_test)[:, 1]
        
        cb_model = cb.CatBoostClassifier()
        cb_model.load_model(f'./model/cb_{i}.cbm')
        iso_cb = joblib.load(f'./model/iso_cb_{i}.pkl')
        raw_cb = cb_model.predict_proba(X_test)[:, 1]
        
        final_preds += (iso_lgb.predict(raw_lgb) * 0.35) + (iso_xgb.predict(raw_xgb) * 0.35) + (iso_cb.predict(raw_cb) * 0.30)
    
    sub['control_success'] = final_preds / 5.0
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    sub.to_csv(OUT_PATH, index=False)

if __name__ == "__main__":
    main()
"""

with open('./submission/script.py', 'w', encoding='utf-8') as f:
    f.write(script_code)

zip_path = 'submission.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk('./submission'):
        for file in files:
            file_path = os.path.join(root, file)
            zipf.write(file_path, os.path.relpath(file_path, './submission'))

print("전처리 대공사 완료본 submission.zip 생성 완료.")


In [ ]:
# [부록] 하이퍼파라미터 튜닝 샌드박스
# 제출에 필요한 셀이 아닙니다. 더 좋은 점수를 위해 파라미터 탐색을 하고 싶을 때만 실행하세요.
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

X_t, X_v, y_t, y_v = train_test_split(train[final_features], train['control_success'], test_size=0.2, stratify=train['control_success'], random_state=42)
optuna.logging.set_verbosity(optuna.logging.INFO)

def lgb_objective(trial):
    params = {
        'objective': 'binary',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'max_depth': trial.suggest_int('max_depth', 5, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'n_estimators': 500, 'verbosity': -1, 'random_state': 42
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_t, y_t, eval_set=[(X_v, y_v)], eval_metric='mse', callbacks=[lgb.early_stopping(20, verbose=False)])
    preds = model.predict_proba(X_v)[:, 1]
    return mean_squared_error(y_v, preds)

print("LGBM 탐색 시작...")
lgb_study = optuna.create_study(direction='minimize')
lgb_study.optimize(lgb_objective, n_trials=3)
print(f"최고 성능 LGBM 파라미터: {lgb_study.best_params}")
